# Phase 1.3 — Segment Check
ตรวจ segmentation ด้วยตา: สุ่ม 50 ภาพ แสดง original | mask | cropped
เน้นภาพ D6–D8 (ใบน้ำตาลจัด) และภาพใน issues log

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_DIR     = Path('../data/raw')
CROPPED_DIR = Path('../data/segmented/cropped')
MASKS_DIR   = Path('../data/segmented/masks')
ISSUES_LOG  = Path('../data/segment_issues.csv')

PATTERN = re.compile(r'^(COS|GOK)(\d{2})_([AB])_D(\d+)_([ME])_(top|side)\.jpg$', re.IGNORECASE)
random.seed(42)

In [ ]:
def show_triplet(ax_row, raw_path: Path, title: str = ''):
    """แสดง original | mask | cropped ใน 1 แถว"""
    stem = raw_path.stem
    mask_path    = MASKS_DIR   / f'{stem}_mask.png'
    cropped_path = CROPPED_DIR / f'{stem}_cropped.jpg'

    orig    = cv2.cvtColor(cv2.imread(str(raw_path)),    cv2.COLOR_BGR2RGB)
    mask    = cv2.imread(str(mask_path),    cv2.IMREAD_GRAYSCALE) if mask_path.exists()    else np.zeros((512,512), dtype=np.uint8)
    cropped = cv2.cvtColor(cv2.imread(str(cropped_path)), cv2.COLOR_BGR2RGB) if cropped_path.exists() else np.zeros((512,512,3), dtype=np.uint8)

    for ax, img, lbl in zip(ax_row, [orig, mask, cropped], ['Original', 'Mask', 'Cropped']):
        ax.imshow(img, cmap='gray' if lbl == 'Mask' else None)
        ax.set_title(f'{lbl}\n{title}', fontsize=7)
        ax.axis('off')

## Section 1: สุ่ม 5 ภาพต่อ Day (D0–D8) = 45 ภาพ

In [ ]:
all_files = sorted(RAW_DIR.glob('*.jpg'))
by_day = {}
for f in all_files:
    m = PATTERN.match(f.name)
    if m:
        day = int(m.group(4))
        by_day.setdefault(day, []).append(f)

sampled = []
for day in sorted(by_day):
    sampled += random.sample(by_day[day], min(5, len(by_day[day])))

print(f'Sample size: {len(sampled)} images across {len(by_day)} days')

fig, axes = plt.subplots(len(sampled), 3, figsize=(12, len(sampled) * 2.5))
for i, img_path in enumerate(sampled):
    show_triplet(axes[i], img_path, title=img_path.name)
plt.suptitle('Segment Check — 5 per Day', fontsize=12, y=1.001)
plt.tight_layout()
plt.savefig('../results/segment_check_per_day.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved: results/segment_check_per_day.png')

## Section 2: ภาพที่อยู่ใน issues log

In [ ]:
if ISSUES_LOG.exists():
    issues = pd.read_csv(ISSUES_LOG)
    print(f'Issues found: {len(issues)}')
    display(issues)

    issue_files = [RAW_DIR / row['filename'] for _, row in issues.iterrows() if (RAW_DIR / row['filename']).exists()]
    if issue_files:
        fig, axes = plt.subplots(len(issue_files), 3, figsize=(12, len(issue_files) * 2.5))
        if len(issue_files) == 1:
            axes = [axes]
        for i, img_path in enumerate(issue_files):
            row = issues[issues['filename'] == img_path.name].iloc[0]
            show_triplet(axes[i], img_path, title=f"{img_path.name}\n[{row['issue']}]")
        plt.suptitle('Segment Issues', fontsize=12)
        plt.tight_layout()
        plt.savefig('../results/segment_check_issues.png', dpi=80, bbox_inches='tight')
        plt.show()
else:
    print('No issues log found — run segment_all first')

## Section 3: สรุป — บันทึกผลการตรวจตา
กรอก ✅ หรือ ❌ หลังดูภาพแต่ละ section แล้ว

In [ ]:
# กรอกผลการตรวจตา
review_notes = {
    'total_checked': len(sampled),
    'visually_correct': None,   # กรอก: จำนวนที่ตัดถูก
    'accuracy_pct': None,       # คำนวณเอง: correct/total*100
    'notes': '',                # สังเกตเห็นอะไรพิเศษ
}
print('กรอก review_notes แล้ว commit ครับ')
print(review_notes)